# Lab 1a · Un agente desde cero (sin frameworks)

**Objetivo:** entender qué hay *dentro* de un agente programándolo a mano, como lo haría un framework:
**modelo + herramientas + bucle**… y **medirlo**.

```
pregunta ──► MODELO ──¿pide una herramienta?──► sí ──► tu código la ejecuta ──► resultado ──┐
               ▲                                                                           │
               └───────────────────────────────────────────────────────────────────────────┘
                                     no ──► respuesta final
```

Las celdas con **✏️ TODO** son para ti. Cada ejercicio tiene una **comprobación automática** (✅/❌), una pista
(`mucdia.lab.pista(...)`) y, si te atascas, la solución (`mucdia.lab.solucion(...)`).

⏱️ 22 minutos. Camino mínimo: pasos **1, 3, 5 y 6**. Si vas rápido: paso 7 y ⭐.

In [ ]:
import inspect
import json
import time
import typing

import numpy as np
import pandas as pd

import mucdia
from mucdia import negocio

CLIENTE = mucdia.cliente_openai()  # misma interfaz que el SDK oficial de OpenAI, apuntando a Gemini
MODELO = mucdia.config.modelo()
print("Modelo:", MODELO)

## Paso 1 · Un modelo sin herramientas… y sin certezas

**a) ¿Sabe el modelo cuánto vendimos?** Preguntemos algo que solo se puede responder con nuestros datos.

In [ ]:
pregunta = "¿Cuánto vendió MUCDIA en septiembre de 2026? Dame la cifra exacta."
respuesta = CLIENTE.chat.completions.create(model=MODELO, messages=[{"role": "user", "content": pregunta}])
print(respuesta.choices[0].message.content)

O se niega, o se inventa una cifra (*alucina*): el modelo **no tiene acceso a nuestros datos**.

**b) Un LLM es un clasificador gigante.** En cada paso calcula unos *logits* para todo su vocabulario
(≈250.000 tokens), los convierte en probabilidades con un **softmax** —el mismo de vuestras redes de
clasificación— y **muestrea** el siguiente token. La **temperatura** *T* divide los logits antes del softmax:

In [ ]:
logits = np.array([3.1, 2.4, 1.0, 0.2])          # ilustrativos: "caído", "bajado", "subido", "sido"
for T in (0.2, 1.0, 2.0):
    p = np.exp(logits / T) / np.exp(logits / T).sum()
    print(f"T = {T}: {np.round(p, 2)}")

Con *T* baja casi siempre sale el token más probable; con *T* alta, la distribución se aplana.
Compruébalo con el modelo de verdad (6 llamadas):

In [ ]:
idea = [{"role": "user", "content": "Inventa un nombre para la nueva marca de café de MUCDIA. Responde solo con el nombre."}]
for temperatura in (0.0, 1.5):
    nombres = [CLIENTE.chat.completions.create(model=MODELO, messages=idea, temperature=temperatura)
               .choices[0].message.content.strip() for _ in range(3)]
    print(f"T = {temperatura}: {nombres}")

🤔 ¿Con **T = 0** salen tres nombres iguales? No tiene por qué: los modelos servidos en GPU no son perfectamente
deterministas (y Gemini 3 está pensado para usarse con T = 1). **Consecuencia para un científico de datos:
una ejecución de un agente no es una medida.** Lo cuantificaremos en la sesión 3.

## Paso 2 · Las herramientas son funciones normales

En `mucdia/negocio.py` hay funciones de Python que consultan la base de datos. Pruébalas directamente:

In [ ]:
print(negocio.ventas_por_tienda("2026-09-01", "2026-09-30"))

## Paso 3 · ✏️ De función a herramienta: el JSON Schema

El modelo no ve tu código: solo ve un **nombre**, una **descripción** y los **parámetros** descritos en
JSON Schema. Con eso decide *cuándo* y *cómo* usar cada herramienta. Los frameworks generan esa descripción
automáticamente a partir de la firma de la función (*type hints*, `Annotated` y *docstring*). Programa tú esa
conversión. Así es la firma de una de nuestras herramientas:

In [ ]:
print(inspect.signature(negocio.top_productos))
print(inspect.getdoc(negocio.top_productos))

In [ ]:
TIPOS = {str: "string", int: "integer", float: "number", bool: "boolean"}


def esquema(funcion):
    """Convierte una función de Python en la descripción JSON que ve el modelo."""
    propiedades, obligatorios = {}, []
    for nombre, p in inspect.signature(funcion).parameters.items():
        tipo, descripcion = p.annotation, ""
        if typing.get_origin(tipo) is typing.Annotated:          # Annotated[str, "descripción"]
            tipo, descripcion = typing.get_args(tipo)[0], typing.get_args(tipo)[1]
        if type(None) in typing.get_args(tipo):                  # str | None  →  str (opcional)
            tipo = next(t for t in typing.get_args(tipo) if t is not type(None))
        propiedades[nombre] = ...   # ✏️ TODO: un diccionario con el "type" (usa TIPOS) y la "description"
        if ...:                     # ✏️ TODO: ¿cuándo es obligatorio un parámetro? (pista: p.default)
            obligatorios.append(nombre)
    return {"type": "function", "function": {
        "name": funcion.__name__,
        "description": ...,         # ✏️ TODO: la descripción de la herramienta es su docstring
        "parameters": {"type": "object", "properties": propiedades, "required": obligatorios},
    }}

In [ ]:
mucdia.lab.comprobar_esquema(esquema)

In [ ]:
FUNCIONES = {f.__name__: f for f in (negocio.ventas_por_categoria, negocio.ventas_por_tienda, negocio.top_productos)}
HERRAMIENTAS = [esquema(f) for f in FUNCIONES.values()]
print("Herramientas:", [h["function"]["name"] for h in HERRAMIENTAS])

## Paso 4 · El modelo *pide* una herramienta… y cada descripción cuesta

In [ ]:
SISTEMA = """Eres el analista de datos de MUCDIA S.A. (una cadena ficticia de tiendas).
Usa las herramientas para obtener cifras reales: nunca te inventes datos.
Hay datos del 2025-01-01 al 2026-10-31. Responde en español, de forma breve y con cifras."""

mensajes = [
    {"role": "system", "content": SISTEMA},
    {"role": "user", "content": "¿Cuáles fueron los 3 productos más vendidos durante el Black Friday 2025 (21 nov - 1 dic)?"},
]
respuesta = CLIENTE.chat.completions.create(model=MODELO, messages=mensajes, tools=HERRAMIENTAS)
mensaje = respuesta.choices[0].message
print("Texto del modelo:", mensaje.content)
for llamada in mensaje.tool_calls or []:
    print("🔧 Pide ejecutar:", llamada.function.name, llamada.function.arguments)

sin_herramientas = CLIENTE.chat.completions.create(model=MODELO, messages=mensajes)
print(f"\nTokens de entrada con herramientas: {respuesta.usage.prompt_tokens} · "
      f"sin herramientas: {sin_herramientas.usage.prompt_tokens}")

1. El modelo devuelve una **intención**: el nombre de la herramienta y sus argumentos en JSON.
   **Quien ejecuta es tu código**, y ahí se ponen los controles (sesión 3).
2. Las descripciones de las herramientas viajan **en cada llamada**: son tokens de entrada que pagas siempre.
   Diez herramientas con descripciones largas encarecen cada paso del agente.

## Paso 5 · ✏️ El bucle agéntico, con telemetría

Completa la función: **pensar** (llamar al modelo) → **medir** → **¿ha terminado?** → **actuar** (ejecutar
herramientas) → **observar** (devolverle los resultados) → repetir. Devuelve la respuesta **y una traza** con
una fila por llamada al modelo.

> 💡 Guarda el mensaje del modelo **tal cual** (`mensajes.append(mensaje)`): Gemini 3 incluye una *firma de
> razonamiento* que necesita en la siguiente vuelta. Y si una herramienta falla, **no rompas el bucle**:
> devuélvele el error al modelo como observación; muchas veces se corrige solo.

In [ ]:
def ejecutar_agente(pregunta, max_pasos=6, cliente=CLIENTE, verbose=True):
    """Devuelve (respuesta_final, traza): la traza tiene una fila por cada llamada al modelo."""
    mensajes = [{"role": "system", "content": SISTEMA}, {"role": "user", "content": pregunta}]
    traza = []
    for paso in range(1, max_pasos + 1):
        # 1) PENSAR. ✏️ TODO: llama al modelo con MODELO, mensajes y HERRAMIENTAS (como en el paso 4)
        t0 = time.perf_counter()
        respuesta = ...
        mensaje = respuesta.choices[0].message

        # 2) MEDIR. ✏️ TODO: tokens de entrada y de salida de esta llamada (están en respuesta.usage)
        traza.append({"paso": paso,
                      "tokens_entrada": ...,
                      "tokens_salida": ...,
                      "ms": round((time.perf_counter() - t0) * 1000),
                      "herramientas": [ll.function.name for ll in (mensaje.tool_calls or [])]})

        # 3) ¿HA TERMINADO? ✏️ TODO: si no pide herramientas (mensaje.tool_calls vacío), devuelve su texto y la traza
        ...

        # 4) Guardamos su mensaje tal cual (incluye la firma de razonamiento de Gemini 3)
        mensajes.append(mensaje)

        # 5) ACTUAR y OBSERVAR. ✏️ TODO: ejecuta cada herramienta; si falla, el resultado es el texto del error
        for llamada in mensaje.tool_calls:
            nombre = llamada.function.name
            argumentos = json.loads(llamada.function.arguments or "{}")
            if verbose:
                print(f"🔧 Paso {paso}: {nombre}({argumentos})")
            resultado = ...  # ✏️ TODO: la función correcta de FUNCIONES con **argumentos (¡y si falla?)
            mensajes.append({"role": "tool", "tool_call_id": llamada.id, "content": str(resultado)})
    return "⚠️ He alcanzado el máximo de pasos sin una respuesta final.", traza

**Pruébalo sin gastar cuota.** Antes de gastar llamadas, comprobamos el bucle con un **modelo simulado** que
sigue un guion: pide tres herramientas a la vez (una con un parámetro que no existe y otra que no existe) y
después responde. Así se hacen los tests de software con modelos de lenguaje.

In [ ]:
mucdia.lab.comprobar_bucle(ejecutar_agente)

Ahora, con el modelo de verdad:

In [ ]:
respuesta, traza = ejecutar_agente(
    "¿Qué categoría vendió más en el tercer trimestre de 2026 y qué tienda vendió más de esa categoría? "
    "Compáralo con el tercer trimestre de 2025."
)
print(respuesta)

🎉 Acabas de programar un agente: el modelo decide **qué** herramientas usar, **en qué orden** y **cuándo parar**.
Fíjate en que puede pedir **varias herramientas en el mismo paso** (en paralelo).

## Paso 6 · Mide tu agente

In [ ]:
df = pd.DataFrame(traza).set_index("paso")
PRECIO_ENTRADA, PRECIO_SALIDA = 0.30, 2.50       # USD por millón de tokens (Flash-Lite, plan de pago)
coste = (df.tokens_entrada.sum() * PRECIO_ENTRADA + df.tokens_salida.sum() * PRECIO_SALIDA) / 1e6
es = lambda x, decimales=0: f"{x:,.{decimales}f}".replace(",", " ").replace(".", ",").replace(" ", ".")  # 3.270 · 0,00205
print(f"{len(df)} llamadas · {es(df.tokens_entrada.sum())} tokens de entrada · {es(df.tokens_salida.sum())} de salida · "
      f"{es(coste, 5)} $ por pregunta · {es(coste * 10_000 * 30)} $ al mes con 10.000 preguntas al día")
ax = df.tokens_entrada.plot.bar(title="Tokens de entrada en cada llamada al modelo", rot=0, figsize=(6, 3))
ax.set_xlabel("paso del bucle")
df

🤔 **¿Por qué crecen los tokens de entrada en cada paso?** Porque en cada llamada reenvías **todo el
historial**: instrucciones, descripciones de herramientas, la pregunta y todos los resultados anteriores.
El coste de un agente crece más rápido que el número de pasos: por eso un modelo con un precio por token más
bajo puede salir **más caro** si necesita más pasos o «piensa» más (lo veremos en clase con datos).

## Paso 7 · Rómpelo (para entenderlo)

**a) Pregunta ambigua.** ¿Qué supone el agente? ¿Pregunta o se lanza?

In [ ]:
print(ejecutar_agente("¿Qué tal fue el verano?")[0])

**b) Una herramienta que falla.** Simulamos una caída de la base de datos. ¿Se recupera el agente? ¿Es honesto?

In [ ]:
original = FUNCIONES["ventas_por_categoria"]
FUNCIONES["ventas_por_categoria"] = lambda **kwargs: "ERROR: la base de datos no responde (timeout)."
print(ejecutar_agente("¿Qué categoría vendió más en octubre de 2026?")[0])
FUNCIONES["ventas_por_categoria"] = original

**c) Una descripción engañosa.** El modelo solo «ve» las descripciones. ¿Qué pasa si mienten?

In [ ]:
real = HERRAMIENTAS[1]["function"]["description"]
HERRAMIENTAS[1]["function"]["description"] = "Devuelve la previsión meteorológica de cada tienda."
print(ejecutar_agente("¿Qué tienda vendió más en octubre de 2026?")[0])
HERRAMIENTAS[1]["function"]["description"] = real

**d) Sin margen para pensar.** ¿Qué ocurre con un solo paso?

In [ ]:
print(ejecutar_agente("Compara las ventas de Hogar en Andalucía en el T3 de 2025 y el T3 de 2026.", max_pasos=1)[0])

## ⭐ Para ir más allá: ¿da siempre la misma respuesta?

Ejecuta 5 veces la pregunta ambigua (≈15 llamadas) y compara: ¿qué periodo entiende por «verano»? ¿Cuántos
pasos da? Si fueras a poner este agente en producción, ¿cómo medirías su fiabilidad?

In [ ]:
filas = []
for i in range(5):
    texto, t = ejecutar_agente("¿Qué tal fue el verano?", verbose=False)
    filas.append({"ejecución": i + 1, "pasos": len(t), "tokens": sum(f["tokens_entrada"] for f in t),
                  "respuesta": texto[:160]})
pd.set_option("display.max_colwidth", 160)
pd.DataFrame(filas).set_index("ejecución")

## Para pensar

1. ¿Quién **decide** qué herramienta usar? ¿Quién la **ejecuta**? ¿Dónde pondrías los controles?
2. Tu bucle sobrevive a los errores de las herramientas. ¿Qué pasaría si una devolviera 10 MB de texto?
3. ¿Cuánto costaría tu agente con 10.000 preguntas al día? ¿Y si cada pregunta necesitara 10 pasos?
4. ¿Qué te falta para producción (límites de tiempo, memoria, trazas, reintentos…)?
   👉 Eso es lo que da un **framework**: lo verás en el **Lab 1b**.

In [ ]:
mucdia.uso()